# Support Vector Machine - Sweeps

## Setup and Imports

In [1]:
import sys

sys.path.append("..")

import wandb
import dotenv

from src.api.run import sweep_svm_ensemble
from src.api.sweep import wandb_sweep

D:\private_git\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
D:\private_git\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: schurtenberger-david (david-schurtenberger) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Experiments

In [4]:
max_runs = 100
sweep_config = {
    "name": "Support Vector Machine Ensemble",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "svm_config": {
            "parameters": {
                "kernel": {"values": ["poly", "rbf", "sigmoid"]},
                "degree": {"distribution": "int_uniform", "min": 2, "max": 6},
                "gamma": {"values": ["scale", "auto"]},
                "tol": {"distribution": "log_uniform_values", "min": 1e-3 / 2, "max": 2e-3},
                "C": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e2},
                "epsilon": {"distribution": "log_uniform_values", "min": 1e-3, "max": 2e-1},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2025},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
                "data_loader": {"value": "season_average_ensemble"},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_svm_ensemble, run_count=max_runs, project="svm")

## Submission from Best Model

In [6]:
from src.dataloaders.ensemble import EnsembleSeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.models.svm import EnsembleSVMRegressorModel, SVMHyperparamConfig
from src.experiments.config import RunConfig
from src.submissions import generate_matchups, create_submission

In [7]:
run = wandb.Api().run("aicomp-mmlm/svm/dt73t2ws")
config = run.config
config

{'run_config': {'data_loader': 'season_average_ensemble',
  'num_features': 97,
  'start_season': 2003,
  'valid_season': 2025},
 'svm_config': {'C': 13.210966394226746,
  'tol': 0.0019114533384128,
  'gamma': 'scale',
  'degree': 6,
  'kernel': 'rbf',
  'epsilon': 0.0040408178870154545}}

In [8]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = EnsembleSeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=97, valid_season=2025, start_season=2003, data_loader='season_average_ensemble')

In [9]:
hyperparameters = SVMHyperparamConfig(**config.get("svm_config", {}))
hyperparameters

SVMHyperparamConfig(kernel='rbf', degree=6, gamma='scale', coef0=0.0, tol=0.0019114533384128, C=13.210966394226746, epsilon=0.0040408178870154545, shrinking=True, cache_size=200, verbose=False, max_iter=-1)

In [10]:
model = EnsembleSVMRegressorModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [11]:
season = 2025
create_submission(season=season, model=model, filename=f"submission_svm_ensemble_{season}.csv", fit=True)

metrics: {'train_brier_ensemble': np.float64(0.17029178867070868)}, step: 2003
metrics: {'valid_brier_ensemble': np.float64(0.17281290010226255)}, step: 2003
metrics: {'train_brier_ensemble': np.float64(0.17018927661816646)}, step: 2004
metrics: {'valid_brier_ensemble': np.float64(0.18083741611833226)}, step: 2004
metrics: {'train_brier_ensemble': np.float64(0.17020998099951262)}, step: 2005
metrics: {'valid_brier_ensemble': np.float64(0.18251592948168022)}, step: 2005
metrics: {'train_brier_ensemble': np.float64(0.1693967298702276)}, step: 2006
metrics: {'valid_brier_ensemble': np.float64(0.20920031569122033)}, step: 2006
metrics: {'train_brier_ensemble': np.float64(0.17097991538542592)}, step: 2007
metrics: {'valid_brier_ensemble': np.float64(0.15623001552754454)}, step: 2007
metrics: {'train_brier_ensemble': np.float64(0.17073312827006495)}, step: 2008
metrics: {'valid_brier_ensemble': np.float64(0.16655269437658324)}, step: 2008
metrics: {'train_brier_ensemble': np.float64(0.170711

WindowsPath('D:/private_git/code/submissions/submission_svm_ensemble_2025.csv')

## Sweeps with Default Features

In [5]:
max_runs = 100
sweep_config = {
    "name": "Support Vector Machine Ensemble (Default Features)",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "svm_config": {
            "parameters": {
                "kernel": {"values": ["poly", "rbf", "sigmoid"]},
                "degree": {"distribution": "int_uniform", "min": 2, "max": 6},
                "gamma": {"values": ["scale", "auto"]},
                "tol": {"distribution": "log_uniform_values", "min": 1e-3 / 2, "max": 2e-3},
                "C": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e2},
                "epsilon": {"distribution": "log_uniform_values", "min": 1e-3, "max": 2e-1},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2025},
                "start_season": {"value": 2003},
                "num_features": {"value": 0},
                "data_loader": {"value": "season_average_ensemble"},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_svm_ensemble, run_count=max_runs, project="svm")

### Submission from Best Model

In [7]:
from src.dataloaders.ensemble import EnsembleSeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.models.svm import EnsembleSVMRegressorModel, SVMHyperparamConfig
from src.experiments.config import RunConfig
from src.submissions import generate_matchups, create_submission

In [11]:
run = wandb.Api().run("685vhxhd")
config = run.config
config

{'run_config': {'num_features': 0, 'start_season': 2003, 'valid_season': 2024},
 'svm_config': {'C': 0.061695606810578424,
  'tol': 0.0005625439224612987,
  'gamma': 'scale',
  'degree': 2,
  'kernel': 'rbf',
  'epsilon': 0.16231707704003384}}

In [12]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = EnsembleSeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=0, valid_season=2024, start_season=2003, data_loader='season_average')

In [13]:
hyperparameters = SVMHyperparamConfig(**config.get("svm_config", {}))
hyperparameters

SVMHyperparamConfig(kernel='rbf', degree=2, gamma='scale', coef0=0.0, tol=0.0005625439224612987, C=0.061695606810578424, epsilon=0.16231707704003384, shrinking=True, cache_size=200, verbose=False, max_iter=-1)

In [14]:
model = EnsembleSVMRegressorModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [15]:
season = 2025
create_submission(
    season=season, model=model, filename=f"submission_svm_ensemble_default_features_{season}.csv", fit=True
)

metrics: {'train_brier': np.float64(0.16245314967507674)}, step: None


WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/submission_svm_default_features_2025.csv')